# NORMALIZACION DE OUTLIERS Y QUIEBRES DE STOCK

**Autor: Abdias Figueredo V.**

**Version: 1.0**

**Agosto 2025**

**Cochabamba, Bolivia**

## Índice
1.  **Introducción**

2.  **Configuración Inicial**

3.  **Analisis Univariado**

4.  **Análisis de Distribuciones**

5.  **Análisis de Outliers**

7.  **Conclusiones**

## 1. INTRODUCCIÓN

Este proyecto tiene como objetivo realizar un Análisis Exploratorio de Datos (EDA) exhaustivo de los datos de ventas, productos e inventario de una empresa importadora de autopartes. La empresa maneja alrededor de 10,000 SKUs diferentes y tiene registros desde 2018 hasta la fecha actual, generados con IA simulando un sistema SAP Business One.

**Objetivos del EDA:**

*   Comprender la estructura y calidad de los datos disponibles.
*   Identificar patrones y tendencias en las ventas a lo largo del tiempo.
*   Analizar el comportamiento de los diferentes SKUs en términos de ventas y rotación de inventario.
*   Detectar posibles problemas en los datos, como valores atípicos o inconsistencias.
*   Generar hipótesis para futuros análisis y modelado predictivo.

**Tablas de Datos:**

*   **OITM (Tabla de Productos):** Contiene información detallada sobre cada producto, incluyendo su código, descripción, precio, etc.
*   **INV1 (Tabla de Ventas):** Registra las transacciones de venta, incluyendo la fecha, el SKU vendido, la cantidad, el precio, etc.
*   **OINM (Tabla de Inventario):** Contiene información sobre los movimientos de inventario, como entradas, salidas y ajustes.

---

## 2. CONFIGURACION INICIAL

### 2.1 Importación de librerías necesarias

In [1]:
# INSTALACION DE REQUIREMENTS
# Comando alternativo: pip install -r requirements.txt
#!pip install -r requirements.txt --quiet
# pip install --only-binary :all: statsforecast

In [2]:
# =============================================================================
# IMPORTACIÓN DE LIBRERÍAS
# =============================================================================
# Manejo y procesamiento de datos
# -----------------------------------------------------------------------------
import numpy as np
import pandas as pd
import os
import scipy.stats as stats
from pandas.tseries.offsets import DateOffset

# Análisis de series temporales
# -----------------------------------------------------------------------------
from statsforecast import StatsForecast
from utilsforecast.plotting import plot_series
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

# Visualización
# -----------------------------------------------------------------------------
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns

# Configuración de advertencias
# -----------------------------------------------------------------------------
import warnings
warnings.filterwarnings("ignore")

# Configuración de estilos de visualización
# =============================================================================
# Configuración general de matplotlib
plt.style.use('classic')
plt.rcParams.update({
    'figure.figsize': (18, 7),
    'axes.facecolor': '#FFFFFF',  # Fondo blanco para mejor legibilidad
    'font.size': 12,
    'axes.titlesize': 14,
    'axes.labelsize': 12,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'legend.fontsize': 10,
    'figure.titlesize': 16
})

# Configuración de seaborn
#sns.set_theme(style="whitegrid", 
#              rc={'axes.facecolor': '#FFFFFF'},
#              font_scale=1.1)

c:\Users\abdia\AppData\Local\Programs\Python\Python313\Lib\site-packages\fs\__init__.py:4: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  __import__("pkg_resources").declare_namespace(__name__)  # type: ignore
c:\Users\abdia\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### 2.2 Carga de datos

In [3]:
# CARGA DE DATOS INV1
dataSales = pd.read_parquet('../data/bronze/inv1.parquet')
dfSales = dataSales.copy()

In [4]:
# CARGA DE DATOS OITM
dataItems = pd.read_parquet('../data/bronze/oitm.parquet')
dfItems = dataItems.copy()

In [ ]:
# CARGA DE DATOS dfWeekStock
dataWeekStock = pd.read_parquet('../data/silver/dfWeekStock.parquet')
dfWeekStock = dataWeekStock.copy()

In [46]:
# CARGA DE DATOS dfABC
dataABC = pd.read_parquet('../data/silver/dfABC.parquet')
dfABC = dataABC.copy()

---

### 2.3 Limpieza inicial

#### 2.3.1 Filtrado de Items

In [6]:
# INNER JOIN ENTRE OINM Y OITM PARA FILTRAR ITEMS INACTIVOS Y GRUPOS
dfSales = dfSales.merge(dfItems[['ItemCode', 'ItemGrp', 'FrozenFor']], 
                        on='ItemCode', 
                        how='inner')

In [7]:
# FILTRAR PARA QUITAR ITEMS INACTIVOS Y GRUPOS
def filter_active_items(df):
    """
    Filtra el DataFrame para eliminar items inactivos y grupos.
    
    Parámetros:
    -----------
    df : pandas.DataFrame
        DataFrame con columnas 'FrozenFor' y 'ItemGrp'
        
    Retorna:
    --------
    pandas.DataFrame
        DataFrame filtrado sin items inactivos ni grupos
    """
    # Filtrar items activos (no congelados) y grupos (no vacíos)
    filtered_df = df[(df['FrozenFor'] == 'N') & (df['ItemGrp'] != 124)]
    
    # Verificar si se eliminaron filas
    if len(filtered_df) < len(df):
        print(f"⚠️ Se eliminaron {len(df) - len(filtered_df):,} filas de items inactivos o grupos vacíos")
    
    return filtered_df

In [8]:
# USO DE LA FUNCION
dfSales = filter_active_items(dfSales)

⚠️ Se eliminaron 76,145 filas de items inactivos o grupos vacíos


In [9]:
# MOSTRAR LOS GRUPOS UNICOS
print("📋 ItemGrp únicos:"
      , dfSales['ItemGrp'].unique())

📋 ItemGrp únicos: [708 101 202 303 405 506 809 607]


#### 2.3.2 Agrupacion por semana

In [18]:
# FUNCION PARA PROCESAR VENTAS SEMANALES

def process_weekly_sales(df):
    """
    Procesa las ventas diarias en un resumen semanal usando isocalendar para semanas ISO 8601,
    considerando solo semanas completas.
    
    Parámetros:
    -----------
    df : pandas.DataFrame
        DataFrame con movimientos diarios de ventas que debe contener las columnas:
        ItemCode, DocDate, Quantity
    
    Retorna:
    --------
    pandas.DataFrame
        Resumen semanal de ventas con las columnas:
        ItemCode, StartWeek, Week, Quantity
    """
    # 1. Preparación de fechas con isocalendar
    df['DocDate'] = pd.to_datetime(df['DocDate'])
    df['IsoYear'] = df['DocDate'].dt.isocalendar().year
    df['IsoWeek'] = df['DocDate'].dt.isocalendar().week
    df['Week'] = df['IsoYear'].astype(str) + '-W' + df['IsoWeek'].astype(str).str.zfill(2)
    df['StartWeek'] = df['DocDate'] - pd.to_timedelta(df['DocDate'].dt.isocalendar().day - 1, unit='D')
    
    # 2. Identificar la última semana completa
    max_date = df['DocDate'].max()
    days_to_end_of_week = 7 - max_date.isocalendar().weekday
    
    if days_to_end_of_week < 7:
        # Usar isocalendar para la última semana completa
        last_complete_date = max_date - pd.Timedelta(days=7)
        last_complete_year = last_complete_date.isocalendar().year
        last_complete_week = last_complete_date.isocalendar().week
        last_complete_week_str = f"{last_complete_year}-W{str(last_complete_week).zfill(2)}"
        df = df[df['Week'] <= last_complete_week_str]
        print(f"⚠️ Se excluyó la semana incompleta: {max_date.isocalendar().year}-W{str(max_date.isocalendar().week).zfill(2)}")
    
    # 3. Agrupar por ítem y semana
    weekly = (df.groupby(['ItemCode', 'Week', 'StartWeek'])
             .agg({
                 'Quantity': 'sum',
                 'LineTotal': 'sum'  # Asegurarse de que LineTotal esté presente
             })
             .reset_index())
    
    # 4. Crear todas las combinaciones de semanas completas
    start_date = weekly['StartWeek'].min()
    end_date = weekly['StartWeek'].max()
    all_weeks = pd.date_range(start=start_date, 
                            end=end_date, 
                            freq='W-MON')
    
    items = weekly['ItemCode'].unique()
    
    # 5. Producto cartesiano de items y semanas
    complete_df = pd.DataFrame(
        [(item, week) for item in items for week in all_weeks],
        columns=['ItemCode', 'StartWeek']
    )
    
    # 6. Añadir formato de semana ISO usando isocalendar
    complete_df['IsoYear'] = complete_df['StartWeek'].dt.isocalendar().year
    complete_df['IsoWeek'] = complete_df['StartWeek'].dt.isocalendar().week
    complete_df['Week'] = complete_df['IsoYear'].astype(str) + '-W' + complete_df['IsoWeek'].astype(str).str.zfill(2)
    
    # 7. Unión con datos existentes
    weekly = pd.merge(
        complete_df,
        weekly[['ItemCode', 'Week', 'Quantity', 'LineTotal']],
        on=['ItemCode', 'Week'],
        how='left'
    )
    
    # 8. Rellenar valores faltantes
    weekly['Quantity'] = weekly['Quantity'].fillna(0)
    weekly['LineTotal'] = weekly['LineTotal'].fillna(0)
    
    # 9. Ordenar y seleccionar columnas finales
    result = weekly[['ItemCode', 'StartWeek', 'Week', 'Quantity', 'LineTotal']]
    
    return result.sort_values(['ItemCode', 'StartWeek'])

In [19]:
# Ejemplo de uso
dfWeekSales = process_weekly_sales(dfSales)
print("Muestra del resumen semanal de ventas:")
display(dfWeekSales.head(10))

⚠️ Se excluyó la semana incompleta: 2025-W32
Muestra del resumen semanal de ventas:


,ItemCode,StartWeek,Week,Quantity,LineTotal
0,SKU-00750,2018-01-01,2018-W01,0.0,0.00
1,SKU-00750,2018-01-08,2018-W02,0.0,0.00
2,SKU-00750,2018-01-15,2018-W03,7.0,2577.89
3,SKU-00750,2018-01-22,2018-W04,0.0,0.00
4,SKU-00750,2018-01-29,2018-W05,0.0,0.00
5,SKU-00750,2018-02-05,2018-W06,0.0,0.00
6,SKU-00750,2018-02-12,2018-W07,0.0,0.00
7,SKU-00750,2018-02-19,2018-W08,0.0,0.00
8,SKU-00750,2018-02-26,2018-W09,0.0,0.00
9,SKU-00750,2018-03-05,2018-W10,0.0,0.00


## 3. NORMALIZACION DE OUTLIERS

In [20]:
def normalize_outliers_by_item(df):
    """
    Detecta y normaliza outliers usando el método IQR por cada item,
    reemplazándolos por la mediana de valores no cero.
    
    Parámetros:
    -----------
    df : pandas.DataFrame
        DataFrame con las columnas ItemCode, StartWeek, Week, Quantity
    
    Retorna:
    --------
    pandas.DataFrame
        DataFrame con outliers normalizados y las mismas columnas de entrada
    """
    def replace_outliers(group):
        # Calcular IQR solo con valores no cero
        non_zero_values = group['Quantity'][group['Quantity'] > 0]
        if len(non_zero_values) == 0:
            return group
            
        Q1 = non_zero_values.quantile(0.25)
        Q3 = non_zero_values.quantile(0.75)
        IQR = Q3 - Q1
        
        # Definir límites
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR
        
        # Identificar outliers
        outliers = (group['Quantity'] > upper_bound) | (group['Quantity'] < lower_bound)
        outliers = outliers & (group['Quantity'] > 0)  # Solo considerar valores no cero
        
        if outliers.any():
            # Calcular mediana de valores no cero
            median_non_zero = non_zero_values.median()
            
            # Reemplazar outliers con la mediana
            group.loc[outliers, 'Quantity'] = median_non_zero
            
            # Imprimir información de outliers encontrados
            n_outliers = outliers.sum()
            if n_outliers > 0:
                print(f"⚠️ Item {group['ItemCode'].iloc[0]}: {n_outliers} outliers reemplazados con mediana = {median_non_zero:.2f}")
        
        return group

    # Procesar cada item
    df_normalized = df.groupby('ItemCode', group_keys=False).apply(replace_outliers)
    
    return df_normalized[['ItemCode', 'StartWeek', 'Week', 'Quantity']].sort_values(['ItemCode', 'StartWeek'])

In [21]:
# Normalizar outliers
dfWeekSalesOutliers = normalize_outliers_by_item(dfWeekSales)

# Mostrar algunos ejemplos
print("\nMuestra del resultado:")
display(dfWeekSalesOutliers.head(10))

⚠️ Item SKU-00750: 2 outliers reemplazados con mediana = 8.00
⚠️ Item SKU-00751: 1 outliers reemplazados con mediana = 8.00
⚠️ Item SKU-00752: 5 outliers reemplazados con mediana = 7.00
⚠️ Item SKU-00754: 3 outliers reemplazados con mediana = 7.00
⚠️ Item SKU-00756: 8 outliers reemplazados con mediana = 7.00
⚠️ Item SKU-00760: 1 outliers reemplazados con mediana = 8.00
⚠️ Item SKU-00761: 3 outliers reemplazados con mediana = 8.00
⚠️ Item SKU-00764: 10 outliers reemplazados con mediana = 6.00
⚠️ Item SKU-00765: 2 outliers reemplazados con mediana = 7.50
⚠️ Item SKU-00766: 9 outliers reemplazados con mediana = 6.00
⚠️ Item SKU-00767: 1 outliers reemplazados con mediana = 8.00
⚠️ Item SKU-00769: 3 outliers reemplazados con mediana = 8.00
⚠️ Item SKU-00771: 1 outliers reemplazados con mediana = 9.00
⚠️ Item SKU-00773: 1 outliers reemplazados con mediana = 8.00
⚠️ Item SKU-00776: 2 outliers reemplazados con mediana = 8.00
⚠️ Item SKU-00777: 6 outliers reemplazados con mediana = 7.00
⚠️ Item

,ItemCode,StartWeek,Week,Quantity
0,SKU-00750,2018-01-01,2018-W01,0.0
1,SKU-00750,2018-01-08,2018-W02,0.0
2,SKU-00750,2018-01-15,2018-W03,7.0
3,SKU-00750,2018-01-22,2018-W04,0.0
4,SKU-00750,2018-01-29,2018-W05,0.0
5,SKU-00750,2018-02-05,2018-W06,0.0
6,SKU-00750,2018-02-12,2018-W07,0.0
7,SKU-00750,2018-02-19,2018-W08,0.0
8,SKU-00750,2018-02-26,2018-W09,0.0
9,SKU-00750,2018-03-05,2018-W10,0.0


## 4. NORMALIZACION DE STOCKOUT

In [32]:
def normalize_stockout_demand(df_sales, df_stock):
    """
    Normaliza la demanda cuando hay stockout (sin ventas y sin stock disponible)
    usando la mediana de ventas no cero por item.
    
    Parámetros:
    -----------
    df_sales : pandas.DataFrame
        DataFrame con las columnas ItemCode, StartWeek, Week, Quantity 
    df_stock : pandas.DataFrame
        DataFrame con las columnas ItemCode, StartWeek, AvailableStock
        
    Retorna:
    --------
    pandas.DataFrame
        DataFrame con la demanda normalizada y columnas adicionales
    """
    # 1. Crear copia del dataframe original
    result = df_sales.copy()
    
    # 2. Merge con datos de stock
    result = result.merge(
        df_stock[['ItemCode', 'StartWeek', 'AvailableStock']], 
        on=['ItemCode', 'StartWeek'], 
        how='left'
    )
    
    # 3. Identificar condición de stockout 
    result['DemandStockout'] = ((result['Quantity'] == 0) & 
                            (result['AvailableStock'] <= 0)).astype(int)
    
    # 4. Normalizar por item usando la mediana de ventas no cero
    def normalize_item(group):
        # Calcular mediana de ventas no cero
        median_sales = group['Quantity'][group['Quantity'] > 0].median()
        
        # Si no hay ventas positivas, usar 0
        if pd.isna(median_sales):
            median_sales = 0
            
        # Crear columna normalizada
        group['Quantity'] = group['Quantity'].copy()
        
        # Reemplazar valores donde DemandStock = 1
        mask = group['DemandStockout'] == 1
        group.loc[mask, 'Quantity'] = median_sales
        
        # Contar normalizaciones
        n_normalized = mask.sum()
        if n_normalized > 0:
            print(f"⚠️ Item {group['ItemCode'].iloc[0]}: {n_normalized} semanas normalizadas con mediana = {median_sales:.2f}")
            
        return group
    
    # 5. Aplicar normalización por item
    result = result.groupby('ItemCode', group_keys=False).apply(normalize_item)
    
    # 6. Ordenar y seleccionar columnas finales
    columns = ['ItemCode', 'StartWeek', 'Week', 'Quantity', 
              'AvailableStock', 'DemandStockout']
    
    return result[columns].sort_values(['ItemCode', 'StartWeek'])

In [33]:
# Normalizar demanda por stockout
dfWeekSalesStockout = normalize_stockout_demand(dfWeekSalesOutliers, dfWeekStock)

# Mostrar resultados
print("\nMuestra del resultado:")
display(dfWeekSalesStockout.head(10))

⚠️ Item SKU-00750: 8 semanas normalizadas con mediana = 8.00
⚠️ Item SKU-00752: 121 semanas normalizadas con mediana = 7.00
⚠️ Item SKU-00756: 42 semanas normalizadas con mediana = 7.00
⚠️ Item SKU-00760: 8 semanas normalizadas con mediana = 8.00
⚠️ Item SKU-00764: 4 semanas normalizadas con mediana = 6.00
⚠️ Item SKU-00765: 4 semanas normalizadas con mediana = 7.25
⚠️ Item SKU-00766: 4 semanas normalizadas con mediana = 6.00
⚠️ Item SKU-00769: 44 semanas normalizadas con mediana = 8.00
⚠️ Item SKU-00771: 14 semanas normalizadas con mediana = 9.00
⚠️ Item SKU-00773: 21 semanas normalizadas con mediana = 8.00
⚠️ Item SKU-00776: 32 semanas normalizadas con mediana = 8.00
⚠️ Item SKU-00777: 22 semanas normalizadas con mediana = 7.00
⚠️ Item SKU-00779: 4 semanas normalizadas con mediana = 7.00
⚠️ Item SKU-00782: 26 semanas normalizadas con mediana = 7.00
⚠️ Item SKU-00783: 1 semanas normalizadas con mediana = 7.00
⚠️ Item SKU-00784: 2 semanas normalizadas con mediana = 6.00
⚠️ Item SKU-007

,ItemCode,StartWeek,Week,Quantity,AvailableStock,DemandStockout
0,SKU-00750,2018-01-01,2018-W01,0.0,102.0,0
1,SKU-00750,2018-01-08,2018-W02,0.0,102.0,0
2,SKU-00750,2018-01-15,2018-W03,7.0,95.0,0
3,SKU-00750,2018-01-22,2018-W04,0.0,95.0,0
4,SKU-00750,2018-01-29,2018-W05,0.0,95.0,0
5,SKU-00750,2018-02-05,2018-W06,0.0,95.0,0
6,SKU-00750,2018-02-12,2018-W07,0.0,95.0,0
7,SKU-00750,2018-02-19,2018-W08,0.0,95.0,0
8,SKU-00750,2018-02-26,2018-W09,0.0,95.0,0
9,SKU-00750,2018-03-05,2018-W10,0.0,95.0,0


## 6. AJUSTES PARA EXPORTAR DATAFRAME

### 6.1 FACT VENTAS SEMANALES

In [49]:
# MERGE DE DATAFRAMES DFWEEKSALES Y DFWEEKSALESSTOCKOUT
FactWeekSales = pd.merge(
    dfWeekSales[['ItemCode', 'StartWeek','Week','Quantity', 'LineTotal']],
    dfWeekSalesStockout[['ItemCode','StartWeek','Week','Quantity','AvailableStock', 'DemandStockout']],
    on=['ItemCode', 'StartWeek', 'Week'],
    how='left'
)

In [50]:
# RENOMBRAR COLUMNAS QUANTITY
FactWeekSales = FactWeekSales.rename(columns={
    'Quantity_y': 'NormalizedQuantity',
    'Quantity_x': 'OriginalQuantity'
})

In [51]:
# AÑADIR COLUMNA PRECIO UNITARIO
FactWeekSales['Price'] = FactWeekSales['LineTotal'] / FactWeekSales['OriginalQuantity']

# RELLENAR LOS VALORES NULOS DEL PRICE CON LOS VALORES PROXIMOS
# Rellenar valores nulos por ItemCode usando forward fill y backward fill
FactWeekSales['Price'] = FactWeekSales.groupby('ItemCode')['Price'].transform(
    lambda x: x.fillna(method='ffill').fillna(method='bfill')
)

# Verificar si quedaron nulos
nulos = FactWeekSales['Price'].isnull().sum()
if nulos > 0:
    print(f"⚠️ Quedaron {nulos} valores nulos en Price")

In [52]:
# ORDENAR COLUMNAS
FactWeekSales = FactWeekSales[['ItemCode', 'StartWeek', 'Week', 
                                       'OriginalQuantity','NormalizedQuantity','Price','LineTotal', 
                                       'AvailableStock', 'DemandStockout']]

## 7. GUARDADO DE DATAFRAME

In [53]:
# GUARDAR EL DATAFRAME PROCESADO DFStock
FactWeekSales.to_parquet('../data/gold/FactWeekSales.parquet', index=False)

## 6. CONCLUSIONES

- Se identificaron que los productos mas vendidos son los SKUs 06074, 03543 y 03063.
- El análisis de distribución mostró que las ventas semanales no siguen una distribución normal (p-valor < 0.05 en la prueba de Shapiro-Wilk), lo que es común en datos de ventas del mundo real debido a factores externos como promociones o eventos estacionales.
- El analisis de outliers mostro que hay 2 valoes atipicos en la variable ventas semanales. Es coveniente normalizar estos datos ya que el proximo paso sera crear un modelo de pronostico.